# Workshop — 2. Zero-Shot Inference with a Local VLA

Here we replace the scripted teacher from Notebook 1 with a small **Vision-Language-Action** model (~450M parameters):

```text
top + wrist images, joint state, instruction
                         ↓
                      SmolVLA
                         ↓
                  6 SO100 actions
```

**SmolVLA is the model.** `LerobotLocalPolicy` is the adapter that makes it a `Policy` compatible with `run_policy()`. A VLA therefore does not replace the policy abstraction: it implements its decision-making component.

> This is a local inference demonstration, not a guarantee of task success. The checkpoint was trained with real SO100 robots and cameras; the MuJoCo images are out of distribution.

## 1. Installation

The runtime and checkpoint are pinned to exact revisions. The first run downloads approximately 0.9 GB of model weights.

In [ ]:
%pip install -q -r requirements.txt

## 2. Imports and Device

On Apple Silicon, `mps` is selected automatically; CPU remains available as a slower fallback.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Video, display
from PIL import Image

sys.path.insert(0, str(Path("code").resolve()))
from vla_pick import (
    INSTRUCTION,
    MODEL_ID,
    MODEL_REVISION,
    VIDEO_PATH,
    build_scene,
    choose_device,
    cube_diagnostics,
    load_policy,
    run_rollout,
)

device = choose_device("auto")
print(f"Modello: {MODEL_ID}@{MODEL_REVISION[:8]}")
print(f"Dispositivo: {device}")
print(f"Istruzione: {INSTRUCTION}")

## 3. Scene and Observations

The model was trained with two image streams: a `top` view and a `wrist` view. The embodiment map in `vla_pick.py` explicitly connects these images to the checkpoint's `camera1` and `camera2` inputs, orders the six joints, and converts between MuJoCo radians and LeRobot degrees/0–100 gripper units.

In [ ]:
sim = build_scene()
observation = sim.get_observation("so100")

display(Image.fromarray(observation["top"]).resize((480, 360)))
display(Image.fromarray(observation["wrist"]).resize((480, 360)))

Inspect both images before continuing: each must contain useful visual information. This check is part of the policy contract, not merely visualization.

## 4. Loading the VLA

The community checkpoint is Apache-2.0 and pinned to the stated commit. `strict_keys=True` prevents unknown camera or state keys from being connected by guesswork.

In [ ]:
policy, device = load_policy(device)
print(type(policy).__name__)
print(f"requires_images={policy.requires_images}")
print(f"execution_horizon={policy.execution_horizon}")

## 5. Rollout

SmolVLA produces chunks of 50 actions. We execute 400 steps at 30 Hz, approximately the average duration of its training demonstrations.

In [ ]:
result = run_rollout(sim, policy, steps=400, video_path=VIDEO_PATH)
print("status:", result["status"])
print(cube_diagnostics(sim))

In [ ]:
display(Video(str(VIDEO_PATH), embed=True, width=640))

## What We Demonstrated

- The VLA really is inside the policy: it uses pixels, language, and state to generate actions.
- `run_policy()` is unchanged from Notebook 1; only the decision-making implementation behind the interface changes.
- `status="success"` means that the rollout executed without errors; it **does not** mean that the task succeeded. Task success is measured with `placed_in_box` and confirmed in the video.
- If the task fails, the most likely cause is not the API but the real-world → MuJoCo domain shift. Notebook 3 will fine-tune SmolVLA on the matching simulation demonstrations collected in Notebook 1.